# 📕 Módulo 05 - Notebook 03: Pivot Tables y Melt

## 🔄 Transformación de estructuras: Ancho ↔ Largo

**Libro:** Saliendo de lo Pandito  
**Módulo:** 05 - Reshaping y Conciliaciones  
**Duración estimada:** 70 minutos  
**Dificultad:** 🟡 Intermedio  
**Plataforma:** Databricks Free Edition

---

## 🎯 Objetivos

✅ **Dominar** pivot_table (largo → ancho)  
✅ **Aplicar** melt (ancho → largo)  
✅ **Entender** cuándo usar cada formato  
✅ **Crear** reportes dinámicos  
✅ **Transformar** datos para visualización

---

## 📚 Contenido

1. Formato Largo vs Ancho
2. pivot_table (Largo → Ancho)
3. melt (Ancho → Largo)
4. Casos de Uso Empresariales
5. Reportes Dinámicos

---

## 💡 Por qué importa

**El mismo dataset, dos formatos:**

* 📈 **Formato Largo:** Ideal para análisis y gráficos
* 📊 **Formato Ancho:** Ideal para reportes y Excel

**Necesitas dominar ambos para:**
* Reportes ejecutivos
* Dashboards
* Análisis comparativos
* Exportación a Excel

In [0]:
import pandas as pd

print("💾 CARGANDO DATOS REALES DESDE UNITY CATALOG")
print("="*70)

CATALOG = "pandito_ds"
SCHEMA = "default"

try:
    df = spark.table(f"{CATALOG}.{SCHEMA}.ventas_mensuales_mendoza_h3").toPandas()
    df['fecha'] = pd.to_datetime(df['fecha'])
    df['mes'] = df['fecha'].dt.month
    df['año'] = df['fecha'].dt.year
    
    print(f"\n✅ Datos reales cargados exitosamente")
    print(f"   📊 Registros: {len(df):,}")
    print(f"   📅 Período: {df['fecha'].min().strftime('%Y-%m-%d')} a {df['fecha'].max().strftime('%Y-%m-%d')}")
    print(f"   🏪 Sucursales: {df['sucursal_id'].nunique()}")
    
    print(f"\n🎯 Listo para pivot_table y melt")
    print(f"   • df: DataFrame con columnas año, mes, sucursal_id, ventas")
    
    USAR_DATOS_REALES = True
    
except Exception as e:
    print(f"\n⚠️  No se pudo cargar la tabla de Unity Catalog")
    print(f"   Error: {e}")
    print(f"\n📝 Solución: Ejecuta primero 00_05_Preparacion_Datos_Empresariales.ipynb")
    print(f"\n   Continuando con datos sintéticos...")
    
    df = None
    USAR_DATOS_REALES = False

print("\n" + "="*70)

## 📚 Formato Largo vs Ancho: ¿Cuál usar?

### 🔄 Los dos formatos

**El mismo dataset puede representarse de dos formas:**

#### 1️⃣ **Formato LARGO (Tidy Data)**

```
Año   Mes   Sucursal   Ventas
2023   1     Centro     1500
2023   1     Norte      2200
2023   2     Centro     1800
2023   2     Norte      2400
```

✅ **Ventajas:**
* Ideal para análisis (groupby, filtros)
* Compatible con bibliotecas de visualización (seaborn, plotly)
* Más fácil agregar nuevas categorías

---

#### 2️⃣ **Formato ANCHO (Pivotado)**

```
Año   Mes   Centro   Norte
2023   1     1500     2200
2023   2     1800     2400
```

✅ **Ventajas:**
* Ideal para reportes ejecutivos
* Fácil de leer (comparaciones rápidas)
* Compatible con Excel y presentaciones

---

### 🔧 Las herramientas

| Operación | Método | Dirección |
|-----------|---------|------------|
| **pivot_table** | `df.pivot_table()` | Largo → Ancho |
| **melt** | `df.melt()` | Ancho → Largo |

---

### 💼 Caso de uso empresarial

**Escenario:** Tienes ventas mensuales por sucursal en formato largo

**Necesitas:**
1. **Reporte ejecutivo** (formato ancho) → Usa `pivot_table`
2. **Dashboard interactivo** (formato largo) → Usa `melt`

**Regla de oro:**
* 📊 **Análisis y gráficos** → Formato **largo**
* 📑 **Reportes y Excel** → Formato **ancho**

In [0]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

print("🔄 PIVOT TABLES Y MELT")
print("="*70)

print(f"\nVersión de Pandas: {pd.__version__}")
print(f"Versión de NumPy: {np.__version__}")

print("\n🎯 En este notebook aprenderás:")
print("  • pivot_table: Formato largo → ancho (filas a columnas)")
print("  • melt: Formato ancho → largo (columnas a filas)")
print("  • Cuándo usar cada formato según tu objetivo")

print("\n📖 Métodos clave:")
print("  - df.pivot_table(values=, index=, columns=, aggfunc=)")
print("  - df.melt(id_vars=, value_vars=, var_name=, value_name=)")

print("\n" + "="*70)
print("✅ Librerías cargadas correctamente")

## 🔄 pivot_table: De largo a ancho con datos reales

### 📊 Nuestro dataset en formato largo

Los datos de **Los Andes Market** vienen en formato largo: cada fila es una observación (un mes, una sucursal, un valor de ventas).

```
fecha       sucursal_id  sucursal_nombre           zona               ventas
2019-01-01  SUC001       Centro - San Martín      Centro Comercial   169804
2019-01-01  SUC002       Las Heras                Zona Residencial    96629
2019-01-01  SUC003       Guaymallén - Av. S. M.  Corredor Comercial  140559
...
```

**Pregunta de negocio:** ¿Cómo comparar las sucursales lado a lado por mes?

---

### 🛠️ Sintaxis de pivot_table

```python
df.pivot_table(
    values='ventas',          # Qué columna agregar
    index='mes',              # Filas (categorías de agrupación)
    columns='sucursal_nombre', # Columnas (categorías a comparar)
    aggfunc='sum'             # Cómo agregar (sum, mean, count)
)
```

**Resultado:** Una tabla donde cada sucursal es una columna y cada mes es una fila.

---

### ⚠️ Parámetros clave

* `values`: la columna numérica a agregar (ej: `ventas`)
* `index`: las columnas que serán las filas (ej: `mes`, `año`)
* `columns`: la columna cuyos valores se convierten en columnas (ej: `sucursal_nombre`)
* `aggfunc`: la función de agregación (`'sum'`, `'mean'`, `'count'`, `'max'`)
* `fill_value`: valor para celdas vacías (ej: `fill_value=0`)
* `margins=True`: añade fila/columna de totales

In [0]:
import pandas as pd

print("🔄 PIVOT_TABLE CON DATOS REALES DE LOS ANDES MARKET")
print("="*70)

if USAR_DATOS_REALES and df is not None:
    print("\n📊 Dataset original (formato LARGO):")
    print(f"   Registros: {len(df):,}")
    print(f"   Columnas: {list(df.columns)}")
    print("\n   Primeras 5 filas:")
    print(df[['fecha', 'sucursal_nombre', 'zona', 'ventas']].head())

    print("\n" + "="*70)
    print("\n1️⃣  PIVOT BÁSICO: Ventas por mes y sucursal")
    print("-"*70)

    pivot_basico = df.pivot_table(
        values='ventas',
        index='mes',
        columns='sucursal_nombre',
        aggfunc='sum'
    )
    print("\n   Tabla pivoteada (formato ANCHO):")
    print(pivot_basico.round(0))
    print(f"\n   Shape: {pivot_basico.shape} (meses × sucursales)")

    print("\n" + "="*70)
    print("\n2️⃣  PIVOT CON AGGFUNC='mean': Promedio mensual")
    print("-"*70)

    pivot_mean = df.pivot_table(
        values='ventas',
        index='mes',
        columns='sucursal_nombre',
        aggfunc='mean'
    )
    print("\n   Promedio de ventas por mes y sucursal:")
    print(pivot_mean.round(0))

    print("\n" + "="*70)
    print("\n3️⃣  PIVOT CON MARGINS (totales)")
    print("-"*70)

    pivot_margins = df.pivot_table(
        values='ventas',
        index='mes',
        columns='sucursal_nombre',
        aggfunc='sum',
        margins=True,
        margins_name='TOTAL'
    )
    print("\n   Ventas con totales (margins=True):")
    print(pivot_margins.round(0))
    print("\n   💡 La fila y columna 'TOTAL' muestran los sumatorios")

    print("\n" + "="*70)
    print("\n4️⃣  PIVOT POR ZONA EN VEZ DE SUCURSAL")
    print("-"*70)

    pivot_zona = df.pivot_table(
        values='ventas',
        index='año',
        columns='zona',
        aggfunc='sum',
        fill_value=0
    )
    print("\n   Ventas por año y zona:")
    print(pivot_zona.round(0))
    print("\n   💡 fill_value=0 reemplaza NaN por 0 (más limpio para reportes)")
else:
    print("⚠️  No hay datos reales disponibles")

print("\n" + "="*70)

## 🔁 melt: De ancho a largo con datos reales

### 🎯 ¿Por qué revertir un pivot?

El formato ancho es excelente para reportes, pero **los gráficos necesitan formato largo**.

```
Formato ANCHO (pivot):           Formato LARGO (melt):
Mes  Centro  Norte  Sur          Mes  Sucursal  Ventas
1    1500    2200   1800    →     1    Centro    1500
2    1800    2400   2100          1    Norte     2200
                                   1    Sur       1800
                                   2    Centro    1800
                                   ...
```

---

### 🛠️ Sintaxis de melt

```python
df_ancho.melt(
    id_vars=['mes'],          # Columnas que se mantienen (identificadores)
    value_vars=['Centro', 'Norte', 'Sur'],  # Columnas a convertir en filas
    var_name='sucursal',      # Nombre de la nueva columna de categorías
    value_name='ventas'       # Nombre de la nueva columna de valores
)
```

---

### ⚠️ Parámetros clave

* `id_vars`: columnas que se conservan como identificadores (NO se funden)
* `value_vars`: columnas que se convierten de columnas a filas
* `var_name`: nombre de la nueva columna que contiene los nombres de las columnas originales
* `value_name`: nombre de la nueva columna que contiene los valores

---

### 💡 Caso de uso real

Después de crear un reporte ejecutivo con `pivot_table`, necesitas graficar la evolución
de ventas por sucursal. Plotly y seaborn requieren formato largo → usas `melt`.

In [0]:
import pandas as pd

print("🔁 MELT: De ancho a largo con datos reales")
print("="*70)

if USAR_DATOS_REALES and df is not None:
    # Partimos del pivot_table que creamos antes
    pivot = df.pivot_table(
        values='ventas',
        index='mes',
        columns='sucursal_nombre',
        aggfunc='sum'
    ).reset_index()

    print("\n📊 Tabla ANCHA (resultado de pivot_table):")
    print(pivot.round(0).head(12))
    print(f"   Shape: {pivot.shape} (formato ancho)")

    print("\n" + "="*70)
    print("\n1️⃣  MELT BÁSICO: Revertir el pivot")
    print("-"*70)

    df_largo = pivot.melt(
        id_vars=['mes'],
        var_name='sucursal_nombre',
        value_name='ventas'
    )
    print("\n   Tabla LARGA (resultado de melt):")
    print(df_largo.round(0).head(15))
    print(f"\n   Shape: {df_largo.shape} (formato largo)")
    print(f"   ✅ Volvimos al formato largo original")

    print("\n" + "="*70)
    print("\n2️⃣  MELT CON VALUE_VARS ESPECÍFICOS")
    print("-"*70)

    # Seleccionar solo algunas sucursales para el melt
    sucursales_seleccion = ['Centro - San Martín', 'Las Heras']

    df_largo_select = pivot.melt(
        id_vars=['mes'],
        value_vars=sucursales_seleccion,
        var_name='sucursal_nombre',
        value_name='ventas'
    )
    print(f"\n   Melt solo de {len(sucursales_seleccion)} sucursales:")
    print(df_largo_select.round(0).head(10))
    print(f"   Shape: {df_largo_select.shape}")

    print("\n" + "="*70)
    print("\n3️⃣  CICLO COMPLETO: Largo → Ancho → Largo")
    print("-"*70)

    # Original (largo)
    print(f"   1. Formato largo original: {len(df)} filas")

    # Pivot (ancho)
    pivot_ciclo = df.pivot_table(
        values='ventas', index=['año', 'mes'],
        columns='sucursal_nombre', aggfunc='sum'
    )
    print(f"   2. Después de pivot_table: {len(pivot_ciclo)} filas (ancho)")

    # Melt (largo de vuelta)
    df_restaurado = pivot_ciclo.reset_index().melt(
        id_vars=['año', 'mes'],
        var_name='sucursal_nombre',
        value_name='ventas'
    )
    print(f"   3. Después de melt: {len(df_restaurado)} filas (largo)")
    print("\n   ✅ El ciclo pivot → melt preserva la información")
else:
    print("⚠️  No hay datos reales disponibles")

print("\n" + "="*70)

## 📊 pivot_table avanzado: Multi-índice y múltiples agregaciones

### 🏗️ Multi-índice en pivot_table

Puedes usar múltiples columnas como `index` o `columns` para crear jerarquías:

```python
df.pivot_table(
    values='ventas',
    index=['año', 'mes'],          # Multi-índice de filas
    columns='sucursal_nombre',     # Columnas simples
    aggfunc='sum'
)
```

**Resultado:**
```
           Centro  Las Heras  Guaymallén
2019  1    169804     96629     140559
      2    166149    105216     141157
      3    ...
2020  1    ...
```

---

### 📈 Múltiples agregaciones

Puedes aplicar varias funciones de agregación al mismo tiempo:

```python
df.pivot_table(
    values='ventas',
    index='sucursal_nombre',
    columns='año',
    aggfunc=['sum', 'mean', 'count']
)
```

**Resultado:**
```
                    sum                          mean
año              2019    2020    2021    2019    2020    2021
Centro           ...     ...     ...     ...     ...     ...
```

---

### 💼 Caso de uso: Reporte ejecutivo anual

Un CFO necesita ver:
1. Ventas totales por sucursal y año → multi-índice + aggfunc='sum'
2. Ranking de sucursales → ordenar por total descendente
3. Variación interanual → calcular pct_change sobre el pivot

In [0]:
import pandas as pd

print("📊 PIVOT_TABLE AVANZADO CON DATOS REALES")
print("="*70)

if USAR_DATOS_REALES and df is not None:
    print("\n1️⃣  MULTI-ÍNDICE: Ventas por año y mes")
    print("-"*70)

    pivot_multi = df.pivot_table(
        values='ventas',
        index=['año', 'mes'],
        columns='sucursal_nombre',
        aggfunc='sum',
        fill_value=0
    )
    print("\n   Pivot con multi-índice (año, mes):")
    print(pivot_multi.round(0).head(15))
    print(f"\n   Shape: {pivot_multi.shape} (años×meses × sucursales)")

    print("\n" + "="*70)
    print("\n2️⃣  MÚLTIPLES AGREGACIONES")
    print("-"*70)

    pivot_multi_agg = df.pivot_table(
        values='ventas',
        index='sucursal_nombre',
        columns='año',
        aggfunc=['sum', 'mean', 'count']
    )
    print("\n   Pivot con sum, mean y count por sucursal y año:")
    print(pivot_multi_agg.round(0))

    print("\n" + "="*70)
    print("\n3️⃣  RANKING DE SUCURSALES POR VENTAS TOTALES")
    print("-"*70)

    # Pivot por sucursal y año, luego ranking
    pivot_ranking = df.pivot_table(
        values='ventas',
        index='sucursal_nombre',
        columns='año',
        aggfunc='sum',
        fill_value=0
    )
    pivot_ranking['TOTAL'] = pivot_ranking.sum(axis=1)
    pivot_ranking = pivot_ranking.sort_values('TOTAL', ascending=False)
    print("\n   Ranking de sucursales por ventas totales:")
    print(pivot_ranking.round(0))

    print("\n" + "="*70)
    print("\n4️⃣  VARIACIÓN INTERANUAL (pct_change)")
    print("-"*70)

    pivot_variacion = df.pivot_table(
        values='ventas',
        index='año',
        columns='sucursal_nombre',
        aggfunc='sum'
    )
    variacion = pivot_variacion.pct_change() * 100
    print("\n   Variación interanual (%):")
    print(variacion.round(1))
    print("\n   💡 Valores positivos = crecimiento, negativos = decrecimiento")

    print("\n" + "="*70)
    print("\n5️⃣  EXPORTAR A EXCEL (formato ancho para stakeholders)")
    print("-"*70)

    reporte_ejecutivo = df.pivot_table(
        values='ventas',
        index='año',
        columns=['zona', 'sucursal_nombre'],
        aggfunc='sum',
        fill_value=0,
        margins=True,
        margins_name='TOTAL'
    )
    print("\n   Reporte ejecutivo (zona → sucursal por año):")
    print(reporte_ejecutivo.round(0))
    print("\n   ✅ Este formato es ideal para exportar a Excel")
else:
    print("⚠️  No hay datos reales disponibles")

print("\n" + "="*70)

## 🏗️ stack() y unstack(): Jerarquías de índices

### 🔄 ¿Qué son stack y unstack?

* **stack()**: Mueve el nivel de columnas al nivel de filas (ancho → largo)
* **unstack()**: Mueve el nivel de filas al nivel de columnas (largo → ancho)

```
unstack()                    stack()
largo ←→ ancho               ancho ←→ largo
```

---

### 📐 Visualización

```
DataFrame ancho:                Después de stack():
       Centro  Norte              sucursal  ventas
mes                              Centro    1500
1      1500   2200      →        Norte     2200
2      1800   2400               mes                    
                                 1  Centro  1500
                                    Norte   2200
                                 2  Centro  1800
                                    Norte   2400
```

---

### 💡 Diferencia con melt/pivot_table

* `melt`: requiere especificar `id_vars` y `value_vars` explícitamente
* `stack()`: opera sobre el MultiIndex automáticamente, más conciso
* `unstack()`: es el equivalente a `pivot_table` pero sobre un MultiIndex existente
* Regla: `stack()` ≈ `melt()`, `unstack()` ≈ `pivot_table()`

In [0]:
import pandas as pd

print("🏗️ STACK Y UNSTACK CON DATOS REALES")
print("="*70)

if USAR_DATOS_REALES and df is not None:
    # Crear un DataFrame con MultiIndex
    df_multi = df.set_index(['año', 'mes', 'sucursal_nombre'])[['ventas']]
    print("\n📊 DataFrame con MultiIndex (año, mes, sucursal):")
    print(df_multi.head(10))

    print("\n" + "="*70)
    print("\n1️⃣  UNSTACK: Sucursal de filas a columnas")
    print("-"*70)

    df_unstacked = df_multi.unstack(level='sucursal_nombre')
    print("\n   Después de unstack('sucursal_nombre'):")
    print(df_unstacked.round(0).head(12))
    print("\n   ✅ Equivalente a pivot_table pero más directo")

    print("\n" + "="*70)
    print("\n2️⃣  STACK: Revertir el unstack")
    print("-"*70)

    df_stacked = df_unstacked.stack()
    print("\n   Después de stack():")
    print(df_stacked.head(10))
    print("\n   ✅ Volvemos al formato largo original")

    print("\n" + "="*70)
    print("\n3️⃣  UNSTACK POR AÑO: Año de filas a columnas")
    print("-"*70)

    df_multi2 = df.set_index(['mes', 'sucursal_nombre', 'año'])[['ventas']]
    df_por_año = df_multi2.unstack(level='año')
    print("\n   Ventas por mes y sucursal, columnas = años:")
    print(df_por_año.round(0).head(12))
    print("\n   💡 Útil para comparar años lado a lado")

    print("\n" + "="*70)
    print("\n4️⃣  UNSTACK ANIDADO: Dos niveles a columnas")
    print("-"*70)

    df_multi3 = df.set_index(['año', 'mes', 'zona', 'sucursal_nombre'])[['ventas']]
    df_anidado = df_multi3.unstack(level=['zona', 'sucursal_nombre'])
    print("\n   Unstack de zona Y sucursal (columnas jerárquicas):")
    print(df_anidado.round(0).head(10))
    print("\n   ✅ Columnas con jerarquía: zona → sucursal_nombre")
else:
    print("⚠️  No hay datos reales disponibles")

print("\n" + "="*70)

## 🎓 Conclusiones del notebook 05_03

### ✅ Lo que aprendiste

1. **Formato Largo (Tidy Data):**
   - Una observación por fila, una variable por columna
   - Ideal para `groupby`, filtros y visualización
   - Compatible con seaborn, plotly y pandas plotting

2. **pivot_table (Largo → Ancho):**
   - `df.pivot_table(values=, index=, columns=, aggfunc=)`
   - Convierte filas en columnas para reportes
   - `aggfunc='sum'` para totales, `'mean'` para promedios

3. **melt (Ancho → Largo):**
   - `df.melt(id_vars=, value_vars=, var_name=, value_name=)`
   - Convierte columnas en filas para análisis
   - Revierte un pivot_table cuando es necesario

4. **Casos de uso empresariales:**
   - Reportes ejecutivos en formato ancho (legibles)
   - Dashboards interactivos en formato largo (gráficos)
   - Exportación a Excel con la estructura correcta

5. **Reportes dinámicos:**
   - Combinar pivot_table + melt para transformaciones flexibles
   - Multi-index pivots para análisis multidimensional
   - Stacking/unstacking para jerarquías

---

### 🎯 Reglas de Oro

👉 **Regla #1: Análisis → Largo, Reportes → Ancho**
```python
# Análisis y gráficos → formato largo
df_largo = df_wide.melt(id_vars='Mes', var_name='Sucursal', value_name='Ventas')

# Reportes y Excel → formato ancho
df_ancho = df.pivot_table(values='Ventas', index='Mes', columns='Sucursal', aggfunc='sum')
```

👉 **Regla #2: Siempre especificar aggfunc en pivot_table**
```python
# MALO: aggfunc por defecto es mean (puede confundir)
df.pivot_table(values='Ventas', index='Mes', columns='Sucursal')

# BUENO: explicitar la agregación
df.pivot_table(values='Ventas', index='Mes', columns='Sucursal', aggfunc='sum')
```

👉 **Regla #3: melt necesita id_vars para no perder columnas**
```python
# MALO: sin id_vars, todo se convierte en filas
df.melt()

# BUENO: preservar columnas identificadoras
df.melt(id_vars=['Año', 'Mes'], value_vars=['Centro', 'Norte'],
        var_name='Sucursal', value_name='Ventas')
```

---

### 📊 Guía de Decisión

| Situación | Método |
|-----------|--------|
| Reporte ejecutivo (comparar categorías) | `pivot_table` → formato ancho |
| Dashboard interactivo (gráficos) | `melt` → formato largo |
| Agregar datos por categoría | `pivot_table(aggfunc='sum')` |
| Promedio por grupo | `pivot_table(aggfunc='mean')` |
| Revertir un pivot | `melt` sobre el resultado ancho |
| Jerarquía de índices | `stack()` / `unstack()` |
| Exportar a Excel legible | `pivot_table` → formato ancho |

---

<div style="background: linear-gradient(90deg, #2563eb 0%, #60a5fa 100%); padding: 20px; border-radius: 10px; color: white; text-align: center;">
  <h3>🔄 ¡Pivot Tables y Melt dominados!</h3>
  <p><i>"El formato correcto es la diferencia entre un reporte claro y un análisis imposible."</i></p>
</div>